In [1]:
!pip install -q "rfdetr[train]" pyyaml opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 13.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.2/382.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 92.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 602.7/602.7 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 55.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 91.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━

In [2]:
import os
import sys
import math
import json
import copy
import requests
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from pathlib import Path

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [3]:
DATASET_DIR = "/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1"

NUM_CLASSES = 21

RESOLUTION = 704
BATCH_SIZE = 1
EPOCHS = 30
LR = 1e-4

OUTPUT_DIR = "/kaggle/working/rfdetr_large_ijepa"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Dataset:", DATASET_DIR)
print("Classes:", NUM_CLASSES)
print("Resolution:", RESOLUTION)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Learning rate:", LR)

Dataset: /kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1
Classes: 21
Resolution: 704
Batch size: 1
Epochs: 30
Learning rate: 0.0001


In [4]:
from pathlib import Path

dataset_path = Path(DATASET_DIR)

print("Dataset exists:", dataset_path.exists())

for item in sorted(dataset_path.iterdir()):
    print(item)

Dataset exists: True
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/README.dataset.txt
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/README.roboflow.txt
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/test
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/train
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/valid


In [5]:
yaml_files = list(dataset_path.glob("*.yaml")) + list(dataset_path.glob("*.yml"))

print("YAML files:", yaml_files)

if yaml_files:
    print(yaml_files[0].read_text()[:5000])

YAML files: []


In [6]:
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

train_dir = dataset_path / "train"

if train_dir.exists():
    train_images = [
        p for p in train_dir.rglob("*")
        if p.suffix.lower() in extensions
    ]

    print("Training images:", len(train_images))
else:
    print("Train directory not found.")

Training images: 4353


In [7]:
%cd /kaggle/working

!rm -rf ijepa

!git clone -q https://github.com/facebookresearch/ijepa.git

print("I-JEPA repository downloaded.")

/kaggle/working
I-JEPA repository downloaded.


In [8]:
IJEPAPATH = "/kaggle/working/ijepa"

if IJEPAPATH not in sys.path:
    sys.path.insert(0, IJEPAPATH)

print(IJEPAPATH)

/kaggle/working/ijepa


In [9]:
IJepa_checkpoint = "/kaggle/working/IN1K-vit.h.14-300e.pth.tar"

url = "https://dl.fbaipublicfiles.com/ijepa/IN1K-vit.h.14-300e.pth.tar"

if not os.path.exists(IJepa_checkpoint):

    print("Downloading official I-JEPA checkpoint...")

    r = requests.get(url, stream=True)
    r.raise_for_status()

    with open(IJepa_checkpoint, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Checkpoint:", IJepa_checkpoint)
print(
    "Size:",
    round(os.path.getsize(IJepa_checkpoint) / 1024**3, 2),
    "GB"
)

Checkpoint: /kaggle/working/IN1K-vit.h.14-300e.pth.tar
Size: 9.65 GB


In [10]:
from src.models.vision_transformer import vit_huge

print("Official I-JEPA ViT-H imported.")

Official I-JEPA ViT-H imported.


In [11]:
checkpoint = torch.load(
    IJepa_checkpoint,
    map_location="cpu"
)

print("Checkpoint keys:")
print(checkpoint.keys())

Checkpoint keys:
dict_keys(['encoder', 'predictor', 'opt', 'scaler', 'target_encoder', 'epoch', 'loss', 'batch_size', 'world_size', 'lr'])


In [12]:
from src.models.vision_transformer import vit_huge

ijepa_encoder = vit_huge(
    patch_size=14,
    img_size=[224],
    predictor_embed_dim=384,
    predictor_depth=12,
)

print("I-JEPA encoder created.")
print("Embedding dimension:", ijepa_encoder.embed_dim)
print("Number of blocks:", len(ijepa_encoder.blocks))

I-JEPA encoder created.
Embedding dimension: 1280
Number of blocks: 32


In [13]:
encoder_state = checkpoint["encoder"]

print("Number of encoder parameters:", len(encoder_state))

Number of encoder parameters: 389


In [14]:
clean_state = {}

for key, value in encoder_state.items():

    new_key = key

    if new_key.startswith("module."):
        new_key = new_key[len("module."):]

    if new_key.startswith("encoder."):
        new_key = new_key[len("encoder."):]

    clean_state[new_key] = value

print("Example keys:")
for k in list(clean_state.keys())[:20]:
    print(k)

Example keys:
pos_embed
patch_embed.proj.weight
patch_embed.proj.bias
blocks.0.norm1.weight
blocks.0.norm1.bias
blocks.0.attn.qkv.weight
blocks.0.attn.qkv.bias
blocks.0.attn.proj.weight
blocks.0.attn.proj.bias
blocks.0.norm2.weight
blocks.0.norm2.bias
blocks.0.mlp.fc1.weight
blocks.0.mlp.fc1.bias
blocks.0.mlp.fc2.weight
blocks.0.mlp.fc2.bias
blocks.1.norm1.weight
blocks.1.norm1.bias
blocks.1.attn.qkv.weight
blocks.1.attn.qkv.bias
blocks.1.attn.proj.weight


In [15]:
missing, unexpected = ijepa_encoder.load_state_dict(
    clean_state,
    strict=False
)

print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

if missing:
    print("\nMissing:")
    print(missing[:20])

if unexpected:
    print("\nUnexpected:")
    print(unexpected[:20])

Missing keys: 0
Unexpected keys: 0


In [16]:
for param in ijepa_encoder.parameters():
    param.requires_grad = False

ijepa_encoder.eval()

trainable = sum(
    p.numel()
    for p in ijepa_encoder.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in ijepa_encoder.parameters()
)

print("Total I-JEPA parameters:", total)
print("Trainable I-JEPA parameters:", trainable)

Total I-JEPA parameters: 630762240
Trainable I-JEPA parameters: 0


In [17]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ijepa_encoder = ijepa_encoder.to(device)

dummy = torch.randn(
    1, 3, 224, 224,
    device=device
)

with torch.no_grad():

    output = ijepa_encoder(dummy)

print("Output shape:", output.shape)

Output shape: torch.Size([1, 256, 1280])


In [18]:
class IJEPAFeatureExtractor(nn.Module):

    def __init__(
        self,
        encoder,
        feature_layers=(8, 16, 24, 32)
    ):
        super().__init__()

        self.encoder = encoder
        self.feature_layers = feature_layers

    @torch.no_grad()
    def forward(self, x):

        # Patch embedding
        x = self.encoder.patch_embed(x)

        # Positional encoding
        x = x + self.encoder.pos_embed

        features = []

        for idx, block in enumerate(self.encoder.blocks, start=1):

            x = block(x)

            if idx in self.feature_layers:

                # [B, N, C]
                B, N, C = x.shape

                side = int(math.sqrt(N))

                feature = x.transpose(1, 2).reshape(
                    B,
                    C,
                    side,
                    side
                )

                features.append(feature)

        return features

In [19]:
ijepa_features = IJEPAFeatureExtractor(
    ijepa_encoder
).to(device)

with torch.no_grad():

    test_features = ijepa_features(dummy)

for i, feature in enumerate(test_features):

    print(
        f"Feature {i}:",
        tuple(feature.shape)
    )

Feature 0: (1, 1280, 16, 16)
Feature 1: (1, 1280, 16, 16)
Feature 2: (1, 1280, 16, 16)
Feature 3: (1, 1280, 16, 16)


In [20]:
from rfdetr import RFDETRLarge

base_model = RFDETRLarge(
    num_classes=NUM_CLASSES,
    resolution=RESOLUTION
)

print(base_model)

[2026-09-15 05:00:47] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-large-2026.pth


/root/.roboflow/models/rf-detr-large-2026.pth:   0%|          | 0.00/130M [00:00<?, ?iB/s]

[2026-09-15 05:00:50] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-large-2026.pth


[2026-09-15 05:00:50] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-15 05:00:50] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-09-15 05:00:51] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-large-2026.pth already exists with correct MD5 hash.


[2026-09-15 05:00:51] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 21. The detection head will be re-initialized to 21 classes.


In [21]:
print("\nBASE MODEL TYPE")
print(type(base_model))
print("\nBASE MODEL ATTRIBUTES")
print([x for x in dir(base_model) if not x.startswith("_")])


BASE MODEL TYPE
<class 'rfdetr.variants.RFDETRLarge'>

BASE MODEL ATTRIBUTES
['callbacks', 'class_names', 'deploy_to_roboflow', 'evaluate', 'export', 'export_for_roboflow', 'from_checkpoint', 'get_model', 'get_model_config', 'get_train_config', 'inference', 'is_optimized_inplace', 'maybe_download_pretrain_weights', 'means', 'model', 'model_config', 'optimize_for_inference', 'predict', 'remove_optimized_model', 'size', 'stds', 'train']


In [22]:
import torch.nn as nn

print("=" * 70)
print("BASE MODEL STRUCTURE")
print("=" * 70)

print("base_model type:")
print(type(base_model))

print("\nAttributes containing 'model', 'module', 'network', 'detr', or 'backbone':")

for name in dir(base_model):
    if any(x in name.lower() for x in
           ["model", "module", "network", "detr", "backbone"]):

        try:
            value = getattr(base_model, name)
            print(
                f"{name:30s} -> "
                f"{type(value).__name__:40s} "
                f"PyTorch={isinstance(value, nn.Module)}"
            )
        except Exception:
            pass

BASE MODEL STRUCTURE
base_model type:
<class 'rfdetr.variants.RFDETRLarge'>

Attributes containing 'model', 'module', 'network', 'detr', or 'backbone':
__module__                     -> str                                      PyTorch=False
_model_config_class            -> ModelMetaclass                           PyTorch=False
get_model                      -> method                                   PyTorch=False
get_model_config               -> method                                   PyTorch=False
model                          -> ModelContext                             PyTorch=False
model_config                   -> RFDETRLargeConfig                        PyTorch=False
remove_optimized_model         -> method                                   PyTorch=False


In [23]:
print("\n" + "=" * 70)
print("VARS(base_model)")
print("=" * 70)

for name, value in vars(base_model).items():

    print(
        f"{name:30s} -> "
        f"{type(value).__name__:40s} "
        f"PyTorch={isinstance(value, nn.Module)}"
    )


VARS(base_model)
model_config                   -> RFDETRLargeConfig                        PyTorch=False
model                          -> ModelContext                             PyTorch=False
callbacks                      -> defaultdict                              PyTorch=False
means                          -> list                                     PyTorch=False
stds                           -> list                                     PyTorch=False
_is_optimized_for_inference    -> bool                                     PyTorch=False
_has_warned_about_not_being_optimized_for_inference -> bool                                     PyTorch=False
_optimized_has_been_compiled   -> bool                                     PyTorch=False
_optimized_batch_size          -> NoneType                                 PyTorch=False
_optimized_resolution          -> NoneType                                 PyTorch=False
_optimized_dtype               -> NoneType                             

In [24]:
print(base_model.model)

In [25]:
import torch
import torch.nn as nn

print("=" * 70)
print("RF-DETR OBJECT INSPECTION")
print("=" * 70)

print("\nbase_model:")
print(type(base_model))

print("\nAttributes of base_model:")
for name in dir(base_model):

    if name.startswith("_"):
        continue

    try:
        value = getattr(base_model, name)

        if isinstance(value, nn.Module):
            print(f"PYTORCH MODULE  : {name} -> {type(value).__name__}")
        elif "model" in name.lower() or "module" in name.lower():
            print(f"OBJECT           : {name} -> {type(value).__name__}")

    except Exception:
        pass

RF-DETR OBJECT INSPECTION

base_model:
<class 'rfdetr.variants.RFDETRLarge'>

Attributes of base_model:
OBJECT           : get_model -> method
OBJECT           : get_model_config -> method
OBJECT           : model -> ModelContext
OBJECT           : model_config -> RFDETRLargeConfig
OBJECT           : remove_optimized_model -> method


In [26]:
print("\n" + "=" * 70)
print("INTERNAL OBJECTS")
print("=" * 70)

for name, value in vars(base_model).items():

    print(
        f"{name:30s} | "
        f"{type(value).__name__:35s} | "
        f"PyTorch Module = {isinstance(value, nn.Module)}"
    )


INTERNAL OBJECTS
model_config                   | RFDETRLargeConfig                   | PyTorch Module = False
model                          | ModelContext                        | PyTorch Module = False
callbacks                      | defaultdict                         | PyTorch Module = False
means                          | list                                | PyTorch Module = False
stds                           | list                                | PyTorch Module = False
_is_optimized_for_inference    | bool                                | PyTorch Module = False
_has_warned_about_not_being_optimized_for_inference | bool                                | PyTorch Module = False
_optimized_has_been_compiled   | bool                                | PyTorch Module = False
_optimized_batch_size          | NoneType                            | PyTorch Module = False
_optimized_resolution          | NoneType                            | PyTorch Module = False
_optimized_dtype     

In [27]:
print("RF-DETR Large configuration:\n")

cfg = base_model.model_config

print("Encoder:", cfg.encoder)
print("Hidden dim:", cfg.hidden_dim)
print("Patch size:", cfg.patch_size)
print("Resolution:", cfg.resolution)
print("Feature indexes:", cfg.out_feature_indexes)
print("Projector scale:", cfg.projector_scale)

RF-DETR Large configuration:

Encoder: dinov2_windowed_small
Hidden dim: 256
Patch size: 16
Resolution: 704
Feature indexes: [3, 6, 9, 12]
Projector scale: ['P4']


In [28]:
class IJEPAAdapter(nn.Module):

    def __init__(
        self,
        in_channels=1280,
        out_channels=384
    ):
        super().__init__()

        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels),
                nn.GELU()
            )
            for _ in range(4)
        ])

    def forward(self, features):

        outputs = []

        for feature, projection in zip(
            features,
            self.projections
        ):

            outputs.append(
                projection(feature)
            )

        return outputs

In [29]:
adapter = IJEPAAdapter(
    in_channels=1280,
    out_channels=384
).to(device)

adapted_features = adapter(test_features)

for i, feature in enumerate(adapted_features):

    print(
        f"Adapted feature {i}:",
        feature.shape
    )

Adapted feature 0: torch.Size([1, 384, 16, 16])
Adapted feature 1: torch.Size([1, 384, 16, 16])
Adapted feature 2: torch.Size([1, 384, 16, 16])
Adapted feature 3: torch.Size([1, 384, 16, 16])


In [30]:
class IJEPAForRFDETR(nn.Module):

    def __init__(
        self,
        encoder,
        feature_layers=(8, 16, 24, 32),
        in_channels=1280,
        out_channels=384
    ):

        super().__init__()

        self.encoder = encoder

        self.feature_layers = feature_layers

        self.adapter = IJEPAAdapter(
            in_channels=in_channels,
            out_channels=out_channels
        )

        # Freeze I-JEPA
        for p in self.encoder.parameters():
            p.requires_grad = False

        self.encoder.eval()

    def forward(self, x):

        features = []

        # I-JEPA
        with torch.no_grad():

            x = self.encoder.patch_embed(x)

            x = x + self.encoder.pos_embed

            for idx, block in enumerate(
                self.encoder.blocks,
                start=1
            ):

                x = block(x)

                if idx in self.feature_layers:

                    B, N, C = x.shape

                    side = int(math.sqrt(N))

                    feature = (
                        x
                        .transpose(1, 2)
                        .reshape(
                            B,
                            C,
                            side,
                            side
                        )
                    )

                    features.append(feature)

        # Adapter
        features = self.adapter(features)

        return features

In [31]:
ijepa_rfdetr_backbone = IJEPAForRFDETR(
    ijepa_encoder,
    feature_layers=(8, 16, 24, 32),
    in_channels=1280,
    out_channels=384
).to(device)

with torch.no_grad():

    features = ijepa_rfdetr_backbone(dummy)

for i, feature in enumerate(features):

    print(
        i,
        feature.shape
    )

0 torch.Size([1, 384, 16, 16])
1 torch.Size([1, 384, 16, 16])
2 torch.Size([1, 384, 16, 16])
3 torch.Size([1, 384, 16, 16])


In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class IJEPA_RFDETR_Backbone(nn.Module):

    def __init__(
        self,
        encoder,
        out_channels=384,
        output_size=44
    ):
        super().__init__()

        self.encoder = encoder

        # I-JEPA is pretrained and will remain frozen
        for param in self.encoder.parameters():
            param.requires_grad = False

        self.encoder.eval()

        self.out_channels = out_channels
        self.output_size = output_size

        # I-JEPA ViT-H has 1280-dimensional embeddings
        self.adapters = nn.ModuleList([
            nn.Conv2d(1280, out_channels, kernel_size=1)
            for _ in range(4)
        ])

        # Use intermediate ViT blocks
        self.feature_indices = [7, 15, 23, 31]


    def _interpolate_pos_embed(self, pos_embed, h, w):

        # Original I-JEPA positional embedding:
        # [1, 256, 1280]
        B, N, C = pos_embed.shape

        original_size = int(N ** 0.5)

        assert original_size * original_size == N, \
            f"Expected square positional embedding, got {N} tokens"

        # [1, 256, 1280]
        #      ↓
        # [1, 1280, 16, 16]
        pos_embed = pos_embed.transpose(1, 2).reshape(
            B,
            C,
            original_size,
            original_size
        )

        # Interpolate to current token grid
        pos_embed = F.interpolate(
            pos_embed,
            size=(h, w),
            mode="bicubic",
            align_corners=False
        )

        # [1, 1280, h, w]
        #      ↓
        # [1, h*w, 1280]
        pos_embed = pos_embed.flatten(2).transpose(1, 2)

        return pos_embed


    def forward(self, x):

        # --------------------------------------------------
        # 1. I-JEPA patch embedding
        # --------------------------------------------------

        with torch.no_grad():

            x = self.encoder.patch_embed(x)

            # Depending on implementation this can be:
            # [B, N, C] or [B, C, H, W]

            if x.ndim == 4:

                B, C, H, W = x.shape

                x = x.flatten(2).transpose(1, 2)

            else:

                B, N, C = x.shape

                H = W = int(N ** 0.5)


            # --------------------------------------------------
            # 2. Resize positional embedding
            # --------------------------------------------------

            pos_embed = self._interpolate_pos_embed(
                self.encoder.pos_embed,
                H,
                W
            )

            x = x + pos_embed


            # --------------------------------------------------
            # 3. Run ViT blocks and collect intermediate features
            # --------------------------------------------------

            features = []

            for idx, block in enumerate(self.encoder.blocks):

                x = block(x)

                if idx in self.feature_indices:

                    feature = x

                    # [B, N, 1280]
                    B, N, C = feature.shape

                    feature = feature.transpose(1, 2).reshape(
                        B,
                        C,
                        H,
                        W
                    )

                    features.append(feature)


        # --------------------------------------------------
        # 4. Adapter: 1280 → 384
        # --------------------------------------------------

        adapted_features = []

        for feature, adapter in zip(
            features,
            self.adapters
        ):

            # Adapter is trainable
            feature = adapter(feature)

            # RF-DETR Large expects approximately
            # 44 × 44 feature maps for 704 resolution
            feature = F.interpolate(
                feature,
                size=(self.output_size, self.output_size),
                mode="bilinear",
                align_corners=False
            )

            adapted_features.append(feature)

        return adapted_features

In [33]:
ijepa_rfdetr_backbone = IJEPA_RFDETR_Backbone(
    encoder=ijepa_encoder,
    out_channels=384,
    output_size=44
)

ijepa_rfdetr_backbone = ijepa_rfdetr_backbone.to(device)

print("I-JEPA → RF-DETR backbone created.")

I-JEPA → RF-DETR backbone created.


In [34]:
dummy_704 = torch.randn(
    1,
    3,
    704,
    704,
    device=device
)

with torch.no_grad():

    features_704 = ijepa_rfdetr_backbone(
        dummy_704
    )

print("\nNumber of features:", len(features_704))

for i, feature in enumerate(features_704):

    print(
        f"Feature {i}: {feature.shape}"
    )


Number of features: 4
Feature 0: torch.Size([1, 384, 44, 44])
Feature 1: torch.Size([1, 384, 44, 44])
Feature 2: torch.Size([1, 384, 44, 44])
Feature 3: torch.Size([1, 384, 44, 44])


In [35]:
if torch.cuda.is_available():

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    print(
        "Allocated:",
        round(allocated, 2),
        "GB"
    )

    print(
        "Reserved:",
        round(reserved, 2),
        "GB"
    )

Allocated: 2.47 GB
Reserved: 3.71 GB


In [36]:
def count_parameters(model):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total, trainable


total, trainable = count_parameters(
    ijepa_rfdetr_backbone
)

print(
    "Total:",
    f"{total:,}"
)

print(
    "Trainable:",
    f"{trainable:,}"
)

Total: 632,729,856
Trainable: 1,967,616


In [37]:
from rfdetr import RFDETRLarge

model = RFDETRLarge(
    num_classes=NUM_CLASSES,
    resolution=RESOLUTION
)

print("RF-DETR Large created.")

[2026-09-15 05:01:57] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-large-2026.pth already exists with correct MD5 hash.


[2026-09-15 05:01:57] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-15 05:01:57] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-09-15 05:01:58] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-large-2026.pth already exists with correct MD5 hash.


[2026-09-15 05:01:59] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 21. The detection head will be re-initialized to 21 classes.


RF-DETR Large created.


In [38]:
print("RF-DETR configuration")
print("----------------------")

cfg = model.model_config

for attr in [
    "encoder",
    "hidden_dim",
    "patch_size",
    "num_windows",
    "out_feature_indexes",
    "projector_scale",
    "resolution",
    "num_classes"
]:

    if hasattr(cfg, attr):

        print(
            attr,
            "=",
            getattr(cfg, attr)
        )

RF-DETR configuration
----------------------
encoder = dinov2_windowed_small
hidden_dim = 256
patch_size = 16
num_windows = 2
out_feature_indexes = [3, 6, 9, 12]
projector_scale = ['P4']
resolution = 704
num_classes = 21


In [39]:
print(model.model)

In [40]:
import torch.nn as nn

print("=" * 70)
print("RF-DETR MODEL INSPECTION")
print("=" * 70)

print("\nmodel type:")
print(type(model))

print("\nmodel.model type:")
print(type(model.model))

print("\nAttributes containing 'model', 'module', 'backbone', or 'net':")

for name in dir(model):

    if name.startswith("_"):
        continue

    try:
        value = getattr(model, name)

        if (
            "model" in name.lower()
            or "module" in name.lower()
            or "backbone" in name.lower()
            or "net" in name.lower()
        ):

            print(
                f"{name:30s} -> "
                f"{type(value).__name__:30s} | "
                f"PyTorch Module: {isinstance(value, nn.Module)}"
            )

    except Exception:
        pass

RF-DETR MODEL INSPECTION

model type:
<class 'rfdetr.variants.RFDETRLarge'>

model.model type:
<class 'rfdetr.inference.ModelContext'>

Attributes containing 'model', 'module', 'backbone', or 'net':
get_model                      -> method                         | PyTorch Module: False
get_model_config               -> method                         | PyTorch Module: False
model                          -> ModelContext                   | PyTorch Module: False
model_config                   -> RFDETRLargeConfig              | PyTorch Module: False
remove_optimized_model         -> method                         | PyTorch Module: False


In [41]:
print("\n" + "=" * 70)
print("model.__dict__")
print("=" * 70)

for name, value in vars(model).items():

    print(
        f"{name:30s} | "
        f"{type(value).__name__:35s} | "
        f"PyTorch Module: {isinstance(value, nn.Module)}"
    )


model.__dict__
model_config                   | RFDETRLargeConfig                   | PyTorch Module: False
model                          | ModelContext                        | PyTorch Module: False
callbacks                      | defaultdict                         | PyTorch Module: False
means                          | list                                | PyTorch Module: False
stds                           | list                                | PyTorch Module: False
_is_optimized_for_inference    | bool                                | PyTorch Module: False
_has_warned_about_not_being_optimized_for_inference | bool                                | PyTorch Module: False
_optimized_has_been_compiled   | bool                                | PyTorch Module: False
_optimized_batch_size          | NoneType                            | PyTorch Module: False
_optimized_resolution          | NoneType                            | PyTorch Module: False
_optimized_dtype               | 

In [42]:
print("=" * 70)
print("MODEL CONTEXT INSPECTION")
print("=" * 70)

print("\nModelContext attributes:")

for name in dir(model.model):

    if name.startswith("_"):
        continue

    try:
        value = getattr(model.model, name)

        print(
            f"{name:35s} -> {type(value).__name__}"
        )

    except Exception:
        pass

MODEL CONTEXT INSPECTION

ModelContext attributes:
args                                -> SimpleNamespace
class_names                         -> NoneType
device                              -> device
inference_model                     -> NoneType
model                               -> LWDETR
postprocess                         -> PostProcess
reinitialize_detection_head         -> method
resolution                          -> int


In [43]:
print("\n" + "=" * 70)
print("MODEL CONTEXT INTERNAL OBJECTS")
print("=" * 70)

for name, value in vars(model.model).items():

    print(
        f"{name:35s} | "
        f"{type(value).__name__:35s} | "
        f"PyTorch Module: {isinstance(value, nn.Module)}"
    )


MODEL CONTEXT INTERNAL OBJECTS
model                               | LWDETR                              | PyTorch Module: True
postprocess                         | PostProcess                         | PyTorch Module: True
device                              | device                              | PyTorch Module: False
resolution                          | int                                 | PyTorch Module: False
args                                | SimpleNamespace                     | PyTorch Module: False
class_names                         | NoneType                            | PyTorch Module: False
inference_model                     | NoneType                            | PyTorch Module: False


In [44]:
import torch.nn as nn

rfdetr_net = model.model.model

print("=" * 70)
print("UNDERLYING RF-DETR NETWORK")
print("=" * 70)

print("Type:", type(rfdetr_net))
print("Is PyTorch Module:", isinstance(rfdetr_net, nn.Module))

UNDERLYING RF-DETR NETWORK
Type: <class 'rfdetr.models.lwdetr.LWDETR'>
Is PyTorch Module: True


In [45]:
print("=" * 70)
print("RF-DETR BACKBONE / PROJECTOR / ENCODER MODULES")
print("=" * 70)

for name, module in rfdetr_net.named_modules():
    if any(k in name.lower() for k in [
        "backbone",
        "projector",
        "encoder"
    ]):
        print(f"{name:70s} -> {type(module).__name__}")

RF-DETR BACKBONE / PROJECTOR / ENCODER MODULES
backbone                                                               -> Joiner
backbone.0                                                             -> Backbone
backbone.0.encoder                                                     -> DinoV2
backbone.0.encoder.encoder                                             -> WindowedDinov2WithRegistersBackbone
backbone.0.encoder.encoder.embeddings                                  -> WindowedDinov2WithRegistersEmbeddings
backbone.0.encoder.encoder.embeddings.patch_embeddings                 -> Dinov2WithRegistersPatchEmbeddings
backbone.0.encoder.encoder.embeddings.patch_embeddings.projection      -> Conv2d
backbone.0.encoder.encoder.embeddings.dropout                          -> Dropout
backbone.0.encoder.encoder.encoder                                     -> WindowedDinov2WithRegistersEncoder
backbone.0.encoder.encoder.encoder.layer                               -> ModuleList
backbone.0.encoder.e

In [46]:
rfdetr_backbone = rfdetr_net.backbone

print("=" * 70)
print("RF-DETR BACKBONE")
print("=" * 70)

print("Type:", type(rfdetr_backbone))
print("Number of children:", len(rfdetr_backbone))

for i, module in enumerate(rfdetr_backbone):
    print(f"\nbackbone[{i}]")
    print("Type:", type(module))
    
    if hasattr(module, "encoder"):
        print("  encoder:", type(module.encoder))
    
    if hasattr(module, "projector"):
        print("  projector:", type(module.projector))

RF-DETR BACKBONE
Type: <class 'rfdetr.models.backbone.Joiner'>
Number of children: 2

backbone[0]
Type: <class 'rfdetr.models.backbone.backbone.Backbone'>
  encoder: <class 'rfdetr.models.backbone.dinov2.DinoV2'>
  projector: <class 'rfdetr.models.backbone.projector.MultiScaleProjector'>

backbone[1]
Type: <class 'rfdetr.models.position_encoding.PositionEmbeddingSine'>


In [47]:
print("=" * 70)
print("DINOv2 ENCODER")
print("=" * 70)

dino = rfdetr_net.backbone[0].encoder

print("Type:", type(dino))
print("\nChildren:")

for name, module in dino.named_children():
    print(f"{name:30s} -> {type(module).__name__}")

DINOv2 ENCODER
Type: <class 'rfdetr.models.backbone.dinov2.DinoV2'>

Children:
encoder                        -> WindowedDinov2WithRegistersBackbone


In [48]:
import inspect

rfdetr_backbone = rfdetr_net.backbone
backbone_block = rfdetr_net.backbone[0]
dino = backbone_block.encoder
projector = backbone_block.projector

print("=" * 70)
print("FORWARD INTERFACE")
print("=" * 70)

print("\n1. Joiner.forward:")
print(inspect.signature(rfdetr_backbone.forward))

print("\n2. Backbone.forward:")
print(inspect.signature(backbone_block.forward))

print("\n3. DinoV2.forward:")
print(inspect.signature(dino.forward))

print("\n4. MultiScaleProjector.forward:")
print(inspect.signature(projector.forward))

FORWARD INTERFACE

1. Joiner.forward:
(tensor_list: rfdetr.utilities.tensors.NestedTensor) -> tuple[list[rfdetr.utilities.tensors.NestedTensor], list[torch.Tensor], list[rfdetr.utilities.tensors.NestedTensor] | None]

2. Backbone.forward:
(tensor_list: 'NestedTensor') -> 'tuple[list[NestedTensor], list[NestedTensor] | None]'

3. DinoV2.forward:
(x: 'Tensor') -> 'list[Tensor]'

4. MultiScaleProjector.forward:
(x: 'list[Tensor]') -> 'list[Tensor]'


In [49]:
# Put the current DINOv2 encoder on the same device as the dummy input
dino = rfdetr_net.backbone[0].encoder.to(device)

dummy_704 = torch.randn(
    1, 3, 704, 704,
    device=device
)

with torch.no_grad():
    dino_features = dino(dummy_704)

print("=" * 70)
print("DINOv2 OUTPUT")
print("=" * 70)

print("Number of features:", len(dino_features))

for i, feature in enumerate(dino_features):
    print(f"Feature {i}: {feature.shape}")

DINOv2 OUTPUT
Number of features: 4
Feature 0: torch.Size([1, 384, 44, 44])
Feature 1: torch.Size([1, 384, 44, 44])
Feature 2: torch.Size([1, 384, 44, 44])
Feature 3: torch.Size([1, 384, 44, 44])


In [50]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class IJEPA_RFDETR_Encoder(nn.Module):
    """
    Pretrained I-JEPA ViT-H/14 encoder adapted for RF-DETR.

    Input:
        [B, 3, H, W]

    Output:
        4 feature maps, each:
        [B, 384, 44, 44]

    I-JEPA:
        1280-dimensional token embeddings

    RF-DETR:
        expects 384-channel multi-scale features
    """

    def __init__(
        self,
        encoder,
        out_channels=384,
        output_size=44,
        feature_indices=(7, 15, 23, 31),
    ):
        super().__init__()

        self.encoder = encoder
        self.out_channels = out_channels
        self.output_size = output_size
        self.feature_indices = feature_indices

        # Freeze pretrained I-JEPA
        for param in self.encoder.parameters():
            param.requires_grad = False

        self.encoder.eval()

        # I-JEPA ViT-H embedding dimension = 1280
        self.adapters = nn.ModuleList([
            nn.Conv2d(
                1280,
                out_channels,
                kernel_size=1
            )
            for _ in feature_indices
        ])

    def interpolate_pos_embed(self, pos_embed, h, w):
        """
        Resize I-JEPA's pretrained positional embeddings
        from its original spatial size to the current token grid.
        """

        B, N, C = pos_embed.shape

        old_size = int(N ** 0.5)

        if old_size * old_size != N:
            raise ValueError(
                f"Expected square positional embedding, "
                f"but received {N} tokens."
            )

        pos_embed = pos_embed.transpose(1, 2)

        pos_embed = pos_embed.reshape(
            B,
            C,
            old_size,
            old_size
        )

        pos_embed = F.interpolate(
            pos_embed,
            size=(h, w),
            mode="bicubic",
            align_corners=False
        )

        pos_embed = pos_embed.flatten(2)
        pos_embed = pos_embed.transpose(1, 2)

        return pos_embed

    def forward(self, x):

        # --------------------------------------------------
        # 1. Frozen I-JEPA encoder
        # --------------------------------------------------
        with torch.no_grad():

            # Patch embedding
            x = self.encoder.patch_embed(x)

            # Convert to [B, N, C]
            if x.ndim == 4:

                B, C, H, W = x.shape

                x = x.flatten(2).transpose(1, 2)

            else:

                B, N, C = x.shape

                H = W = int(N ** 0.5)

            # --------------------------------------------------
            # 2. Interpolate positional embeddings
            # --------------------------------------------------
            pos_embed = self.interpolate_pos_embed(
                self.encoder.pos_embed,
                H,
                W
            )

            x = x + pos_embed

            # --------------------------------------------------
            # 3. Extract intermediate I-JEPA features
            # --------------------------------------------------
            features = []

            for idx, block in enumerate(self.encoder.blocks):

                x = block(x)

                if idx in self.feature_indices:

                    feature = x

                    B, N, C = feature.shape

                    feature = (
                        feature
                        .transpose(1, 2)
                        .reshape(B, C, H, W)
                    )

                    features.append(feature)

        # --------------------------------------------------
        # 4. Adapt I-JEPA 1280 → RF-DETR 384
        # --------------------------------------------------
        adapted_features = []

        for feature, adapter in zip(
            features,
            self.adapters
        ):

            feature = adapter(feature)

            # RF-DETR Large expects 44 × 44
            feature = F.interpolate(
                feature,
                size=(
                    self.output_size,
                    self.output_size
                ),
                mode="bilinear",
                align_corners=False
            )

            adapted_features.append(feature)

        return adapted_features

In [51]:
ijepa_rfdetr_encoder = IJEPA_RFDETR_Encoder(
    encoder=ijepa_encoder,
    out_channels=384,
    output_size=44,
    feature_indices=(7, 15, 23, 31)
).to(device)

print("I-JEPA → RF-DETR adapter created.")

I-JEPA → RF-DETR adapter created.


In [52]:
rfdetr_net.backbone[0].encoder = ijepa_rfdetr_encoder

print("DINOv2 encoder replaced with I-JEPA encoder.")

DINOv2 encoder replaced with I-JEPA encoder.


In [53]:
dummy_704 = torch.randn(
    1, 3, 704, 704,
    device=device
)

with torch.no_grad():
    ijepa_features = rfdetr_net.backbone[0].encoder(dummy_704)

print("=" * 70)
print("I-JEPA → RF-DETR OUTPUT")
print("=" * 70)

print("Number of features:", len(ijepa_features))

for i, feature in enumerate(ijepa_features):
    print(
        f"Feature {i}: "
        f"{feature.shape}"
    )

I-JEPA → RF-DETR OUTPUT
Number of features: 4
Feature 0: torch.Size([1, 384, 44, 44])
Feature 1: torch.Size([1, 384, 44, 44])
Feature 2: torch.Size([1, 384, 44, 44])
Feature 3: torch.Size([1, 384, 44, 44])


In [54]:
# Put the RF-DETR projector on the same device
projector = rfdetr_net.backbone[0].projector.to(device)

# Make sure the RF-DETR network backbone components are on the same device
rfdetr_net.backbone[0] = rfdetr_net.backbone[0].to(device)

print("I-JEPA adapter device:")
print(next(ijepa_rfdetr_encoder.parameters()).device)

print("\nProjector device:")
print(next(projector.parameters()).device)

I-JEPA adapter device:
cuda:0

Projector device:
cuda:0


In [55]:
dummy_704 = torch.randn(
    1, 3, 704, 704,
    device=device
)

with torch.no_grad():
    ijepa_features = rfdetr_net.backbone[0].encoder(dummy_704)

print("=" * 70)
print("I-JEPA FEATURES")
print("=" * 70)

for i, feature in enumerate(ijepa_features):
    print(f"Feature {i}: {feature.shape} | {feature.device}")

I-JEPA FEATURES
Feature 0: torch.Size([1, 384, 44, 44]) | cuda:0
Feature 1: torch.Size([1, 384, 44, 44]) | cuda:0
Feature 2: torch.Size([1, 384, 44, 44]) | cuda:0
Feature 3: torch.Size([1, 384, 44, 44]) | cuda:0


In [56]:
with torch.no_grad():
    projected_features = projector(ijepa_features)

print("=" * 70)
print("MULTISCALE PROJECTOR OUTPUT")
print("=" * 70)

print("Number of features:", len(projected_features))

for i, feature in enumerate(projected_features):
    print(
        f"Feature {i}: "
        f"{feature.shape} | "
        f"{feature.device}"
    )

MULTISCALE PROJECTOR OUTPUT
Number of features: 1
Feature 0: torch.Size([1, 256, 44, 44]) | cuda:0


In [57]:
# ============================================================
# TEST COMPLETE RF-DETR BACKBONE
# I-JEPA → Projector
# ============================================================

# Make sure the complete RF-DETR backbone is on the GPU
rfdetr_net.backbone = rfdetr_net.backbone.to(device)

dummy_704 = torch.randn(
    1, 3, 704, 704,
    device=device
)

# RF-DETR uses NestedTensor
from rfdetr.utilities.tensors import NestedTensor

mask = torch.zeros(
    (1, 704, 704),
    dtype=torch.bool,
    device=device
)

nested_dummy = NestedTensor(
    dummy_704,
    mask
)

print("=" * 70)
print("TESTING COMPLETE RF-DETR BACKBONE")
print("=" * 70)

with torch.no_grad():
    backbone_output = rfdetr_net.backbone(nested_dummy)

print("\nBackbone output type:")
print(type(backbone_output))

print("\nNumber of returned objects:")
print(len(backbone_output))

for i, item in enumerate(backbone_output):
    print(f"\nOutput {i}:")
    print("Type:", type(item))

    if isinstance(item, list):
        print("Number of features:", len(item))

        for j, feature in enumerate(item):
            if hasattr(feature, "tensors"):
                print(
                    f"  Feature {j}: "
                    f"{feature.tensors.shape}"
                )
            else:
                print(
                    f"  Feature {j}: "
                    f"{feature.shape}"
                )

TESTING COMPLETE RF-DETR BACKBONE

Backbone output type:
<class 'tuple'>

Number of returned objects:
3

Output 0:
Type: <class 'list'>
Number of features: 1
  Feature 0: torch.Size([1, 256, 44, 44])

Output 1:
Type: <class 'list'>
Number of features: 1
  Feature 0: torch.Size([1, 256, 44, 44])

Output 2:
Type: <class 'NoneType'>


In [58]:
# ============================================================
# TEST COMPLETE RF-DETR FORWARD
# ============================================================

print("=" * 70)
print("TESTING COMPLETE RF-DETR FORWARD")
print("=" * 70)

print("Model type:", type(model))
print("Network type:", type(rfdetr_net))

# Make sure the whole network is on the same device
rfdetr_net = rfdetr_net.to(device)

dummy_704 = torch.randn(
    1, 3, 704, 704,
    device=device
)

try:
    with torch.no_grad():
        output = rfdetr_net(dummy_704)

    print("\n" + "=" * 70)
    print("SUCCESS!")
    print("=" * 70)

    print("Output type:", type(output))

    if isinstance(output, dict):
        print("\nOutput keys:")
        for key, value in output.items():
            if torch.is_tensor(value):
                print(f"{key}: {value.shape}")
            else:
                print(f"{key}: {type(value)}")
    else:
        print("Output:", output)

except Exception as e:
    print("\n" + "=" * 70)
    print("FORWARD PASS FAILED")
    print("=" * 70)

    print(type(e).__name__, ":", e)

TESTING COMPLETE RF-DETR FORWARD
Model type: <class 'rfdetr.variants.RFDETRLarge'>
Network type: <class 'rfdetr.models.lwdetr.LWDETR'>

SUCCESS!
Output type: <class 'dict'>

Output keys:
pred_logits: torch.Size([1, 3900, 22])
pred_boxes: torch.Size([1, 3900, 4])
aux_outputs: <class 'list'>
enc_outputs: <class 'dict'>


In [59]:
print("=" * 70)
print("TRAINABLE PARAMETER CHECK")
print("=" * 70)

total_params = 0
trainable_params = 0

for name, param in rfdetr_net.named_parameters():

    total_params += param.numel()

    if param.requires_grad:
        trainable_params += param.numel()

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")

TRAINABLE PARAMETER CHECK
Total parameters:     644,071,388
Trainable parameters: 13,309,148
Frozen parameters:    630,762,240


In [60]:
print("\n" + "=" * 70)
print("I-JEPA GRADIENT STATUS")
print("=" * 70)

ijepa_trainable = sum(
    p.numel()
    for p in ijepa_rfdetr_encoder.encoder.parameters()
    if p.requires_grad
)

ijepa_total = sum(
    p.numel()
    for p in ijepa_rfdetr_encoder.encoder.parameters()
)

print(f"I-JEPA total parameters:     {ijepa_total:,}")
print(f"I-JEPA trainable parameters: {ijepa_trainable:,}")


I-JEPA GRADIENT STATUS
I-JEPA total parameters:     630,762,240
I-JEPA trainable parameters: 0


In [61]:
# Keep pretrained I-JEPA frozen and in evaluation mode
ijepa_rfdetr_encoder.encoder.eval()

for p in ijepa_rfdetr_encoder.encoder.parameters():
    p.requires_grad = False

print("I-JEPA frozen:", not any(
    p.requires_grad for p in ijepa_rfdetr_encoder.encoder.parameters()
))

I-JEPA frozen: True


In [62]:
import inspect

print("RF-DETR model:", type(model))
print("Train method:", model.train)
print("Train signature:")
print(inspect.signature(model.train))

print("\nTrain method source file:")
print(inspect.getsourcefile(model.train))

RF-DETR model: <class 'rfdetr.variants.RFDETRLarge'>
Train method: <bound method RFDETR.train of <rfdetr.variants.RFDETRLarge object at 0x7e8db294a6f0>>
Train signature:
(**kwargs: 'Any') -> 'None'

Train method source file:
/usr/local/lib/python3.12/dist-packages/rfdetr/detr.py


In [63]:
# ============================================================
# CELL 60 — KEEP I-JEPA FROZEN DURING RF-DETR TRAINING
# ============================================================

# Make sure the I-JEPA encoder is frozen
for param in ijepa_rfdetr_encoder.encoder.parameters():
    param.requires_grad = False

# Keep I-JEPA in evaluation mode
ijepa_rfdetr_encoder.encoder.eval()

# Verify
ijepa_trainable = sum(
    p.numel()
    for p in ijepa_rfdetr_encoder.encoder.parameters()
    if p.requires_grad
)

print("I-JEPA trainable parameters:", ijepa_trainable)
print("I-JEPA training mode:", ijepa_rfdetr_encoder.encoder.training)

assert ijepa_trainable == 0
assert ijepa_rfdetr_encoder.encoder.training is False

print("\n✓ I-JEPA is frozen and in eval mode.")

I-JEPA trainable parameters: 0
I-JEPA training mode: False

✓ I-JEPA is frozen and in eval mode.


In [64]:
# ============================================================
# CELL 61 — TRAINABLE PARAMETER CHECK
# ============================================================

total_params = 0
trainable_params = 0
ijepa_params = 0

for name, param in rfdetr_net.named_parameters():

    total_params += param.numel()

    if param.requires_grad:
        trainable_params += param.numel()

    if "backbone.0.encoder" in name:
        ijepa_params += param.numel()

print("=" * 70)
print("PARAMETER SUMMARY")
print("=" * 70)

print(f"Total RF-DETR parameters      : {total_params:,}")
print(f"Trainable parameters          : {trainable_params:,}")
print(f"I-JEPA parameters in network : {ijepa_params:,}")

print("=" * 70)

print("\nTrainable parameter groups:")

for name, param in rfdetr_net.named_parameters():
    if param.requires_grad:
        print(f"{name:70s} {param.numel():>12,}")

PARAMETER SUMMARY
Total RF-DETR parameters      : 644,071,388
Trainable parameters          : 13,309,148
I-JEPA parameters in network : 632,729,856

Trainable parameter groups:
transformer.decoder.layers.0.self_attn.in_proj_weight                       196,608
transformer.decoder.layers.0.self_attn.in_proj_bias                             768
transformer.decoder.layers.0.self_attn.out_proj.weight                       65,536
transformer.decoder.layers.0.self_attn.out_proj.bias                            256
transformer.decoder.layers.0.norm1.weight                                       256
transformer.decoder.layers.0.norm1.bias                                         256
transformer.decoder.layers.0.cross_attn.sampling_offsets.weight              16,384
transformer.decoder.layers.0.cross_attn.sampling_offsets.bias                    64
transformer.decoder.layers.0.cross_attn.attention_weights.weight              8,192
transformer.decoder.layers.0.cross_attn.attention_weights.bias     

In [65]:
# ============================================================
# RF-DETR LARGE + I-JEPA
# SAVE ONLY 10 PERIODIC CHECKPOINTS
# ============================================================

DATASET_DIR = "/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1"

OUTPUT_DIR = "/kaggle/working/rfdetr_large_ijepa"

EPOCHS = 20
BATCH_SIZE = 1
LR = 1e-4
RESOLUTION = 704

# 20 epochs / 10 checkpoints = save every 3 epochs
CHECKPOINT_INTERVAL = 10

print("=" * 70)
print("RF-DETR LARGE + I-JEPA TRAINING")
print("=" * 70)

print(f"Dataset              : {DATASET_DIR}")
print(f"Resolution           : {RESOLUTION}")
print(f"Batch size           : {BATCH_SIZE}")
print(f"Epochs               : {EPOCHS}")
print(f"Learning rate        : {LR}")
print(f"Checkpoint interval  : {CHECKPOINT_INTERVAL} epochs")
print(f"Expected checkpoints : {EPOCHS // CHECKPOINT_INTERVAL}")
print("=" * 70)


# ------------------------------------------------------------
# Make absolutely sure I-JEPA is frozen
# ------------------------------------------------------------

for param in ijepa_rfdetr_encoder.encoder.parameters():
    param.requires_grad = False

ijepa_rfdetr_encoder.encoder.eval()


# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

model.train(
    dataset_dir=DATASET_DIR,

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    lr=LR,

    output_dir=OUTPUT_DIR,

    checkpoint_interval=CHECKPOINT_INTERVAL,
)

RF-DETR LARGE + I-JEPA TRAINING
Dataset              : /kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1
Resolution           : 704
Batch size           : 1
Epochs               : 20
Learning rate        : 0.0001
Checkpoint interval  : 10 epochs
Expected checkpoints : 2


[2026-09-15 05:04:20] [WARNING] rf-detr - Dataset '/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1' has 20 classes but model was initialized with num_classes=21. Using the model's configured value (21). If this is unintentional, reinitialize the model with num_classes=20.
[2026-09-15 05:04:20] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-15 05:04:20] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-09-15 05:04:21] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-large-2026.pth already exists with correct MD5 hash.


[2026-09-15 05:04:22] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 21. The detection head will be re-initialized to 21 classes.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[2026-09-15 05:04:38] [INFO] rf-detr - Building Roboflow train dataset with square resize at resolution 704
[2026-09-15 05:04:38] [INFO] rf-detr - Using multi-scale training with square resize and scales: [864]
[2026-09-15 05:04:38] [INFO] rf-detr - Built 1 Albumentations transforms from config
[2026-09-15 05:04:38] [INFO] rf-detr - Built 1 Albumentations transforms from config
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
[2026-09-15 05:04:38] [INFO] rf-detr - Skipping unannotated COCO grouping categories when assigning label indices: 0 (Phase2-Phase3-tomato-Spodop-iEGw)
[2026-09-15 05:04:39] [INFO] rf-detr - Building Roboflow val dataset with square resize at resolution 704
[2026-09-15 05:04:39] [INFO] rf-detr - Using multi-scale training with square resize and scales: [864]
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


...

[2026-09-15 05:36:19] [INFO] rf-detr - Skipping validation-loss computation: compute_val_loss='auto' found no scheduler or callback monitoring 'val/loss', so no 'val/loss' metric is logged this run. Set compute_val_loss=True to log it every validation epoch (for example to read the curve from metrics.csv, TensorBoard, or Weights & Biases).


Output()

[2026-09-15 05:37:27] [INFO] rf-detr - Best EMA metric improved to 0.1261 (epoch 0)
[2026-09-15 06:10:00] [INFO] rf-detr - Best EMA metric improved to 0.1653 (epoch 1)
[2026-09-15 06:43:04] [INFO] rf-detr - Best EMA metric improved to 0.1715 (epoch 2)
[2026-09-15 07:15:47] [INFO] rf-detr - Best EMA metric improved to 0.1976 (epoch 3)
[2026-09-15 07:48:21] [INFO] rf-detr - Best EMA metric improved to 0.2075 (epoch 4)
[2026-09-15 08:20:59] [INFO] rf-detr - Best EMA metric improved to 0.2108 (epoch 5)
[2026-09-15 09:58:42] [INFO] rf-detr - Best EMA metric improved to 0.2219 (epoch 8)
[2026-09-15 10:31:01] [INFO] rf-detr - Best EMA metric improved to 0.2344 (epoch 9)
[2026-09-15 11:03:29] [INFO] rf-detr - Best EMA metric improved to 0.2674 (epoch 10)


`Trainer.fit` stopped: `max_epochs=20` reached.


[2026-09-15 15:52:22] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.0000, ema=0.2674)


In [66]:
# ============================================================
# CHECK RF-DETR EVALUATION API
# ============================================================

import inspect

print("RF-DETR model class:")
print(type(model))

print("\nAvailable evaluation-related methods:")

for name in dir(model):
    if any(x in name.lower() for x in [
        "eval",
        "test",
        "predict",
        "validate",
        "val"
    ]):
        try:
            attr = getattr(model, name)
            if callable(attr):
                print(f"\n{name}")
                print(inspect.signature(attr))
        except Exception:
            pass

RF-DETR model class:
<class 'rfdetr.variants.RFDETRLarge'>

Available evaluation-related methods:

_ensure_eval_mode_for_unoptimized_inference
() -> 'None'

evaluate
(*, split: "Literal['test', 'val']" = 'test', **kwargs: 'Any') -> 'dict[str, float]'

predict
(images: 'str | Image.Image | np.ndarray[Any, Any] | torch.Tensor | list[str | np.ndarray[Any, Any] | Image.Image | torch.Tensor]', threshold: 'float' = 0.5, shape: 'tuple[int, int] | None' = None, patch_size: 'int | None' = None, include_source_image: 'bool' = True, **kwargs: 'Any') -> 'Detections | KeyPoints | list[Detections | KeyPoints]'


In [69]:
# ============================================================
# RF-DETR + I-JEPA — FINAL TEST EVALUATION
# ============================================================

DATASET_DIR = "/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1"

print("=" * 70)
print("RF-DETR + I-JEPA FINAL TEST EVALUATION")
print("=" * 70)

results = model.evaluate(
    split="test",
    dataset_dir=DATASET_DIR
)

print("\nRaw results:")
print(results)

print("\n" + "=" * 70)
print("TEST RESULTS")
print("=" * 70)

for key, value in results.items():
    print(f"{key:30s}: {value:.4f}")

print("=" * 70)

[2026-09-15 15:55:07] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-15 15:55:07] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


RF-DETR + I-JEPA FINAL TEST EVALUATION
[2026-09-15 15:55:08] [INFO] rf-detr - Building Roboflow test dataset with square resize at resolution 704
[2026-09-15 15:55:08] [INFO] rf-detr - Using multi-scale training with square resize and scales: [864]


/tmp/ipykernel_58/3661594295.py:11: UserWarning: Dataset '/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1' has 20 classes but the model has num_classes=21. Evaluating with the model's head unchanged; class indices may not line up with the dataset.
  results = model.evaluate(
Using bfloat16 Automatic Mixed Precision (AMP)


loading annotations into memory...
Done (t=0.02s)
creating index...
index created!


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/rfdetr/models/_assignment.py:76: RuntimeWarning: Triton linear-assignment support is unavailable for this CUDA input; using SciPy on CPU.
  assignment = torch_linear_assignment.batch_linear_assignment(stacked)


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test/F1          │    0.48650485277175903    │
│         test/loss         │     9.381949424743652     │
│        test/mAP_50        │    0.5013025403022766     │
│      test/mAP_50_95       │    0.27787336707115173    │
│        test/mAP_75        │    0.27045994997024536    │
│         test/mAR          │    0.5276862978935242     │
│      test/precision       │    0.5317834615707397     │
│        test/recall        │    0.48023736476898193    │
└───────────────────────────┴───────────────────────────┘


Raw results:
{'test/loss': 9.381949424743652, 'test/mAP_50_95': 0.27787336707115173, 'test/mAP_50': 0.5013025403022766, 'test/mAP_75': 0.27045994997024536, 'test/mAR': 0.5276862978935242, 'test/F1': 0.48650485277175903, 'test/precision': 0.5317834615707397, 'test/recall': 0.48023736476898193}

TEST RESULTS
test/loss                     : 9.3819
test/mAP_50_95                : 0.2779
test/mAP_50                   : 0.5013
test/mAP_75                   : 0.2705
test/mAR                      : 0.5277
test/F1                       : 0.4865
test/precision                : 0.5318
test/recall                   : 0.4802


In [70]:
# ============================================================
# RF-DETR + I-JEPA — TEST EVALUATION
# ============================================================

DATASET_DIR = "/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1"

print("=" * 70)
print("RF-DETR + I-JEPA FINAL TEST EVALUATION")
print("=" * 70)

results = model.evaluate(
    split="test",
    dataset_dir=DATASET_DIR
)

print("\n" + "=" * 70)
print("RAW TEST RESULTS")
print("=" * 70)

print(results)

[2026-09-15 15:55:34] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-15 15:55:34] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


RF-DETR + I-JEPA FINAL TEST EVALUATION
[2026-09-15 15:55:35] [INFO] rf-detr - Building Roboflow test dataset with square resize at resolution 704
[2026-09-15 15:55:35] [INFO] rf-detr - Using multi-scale training with square resize and scales: [864]


/tmp/ipykernel_58/3182126370.py:11: UserWarning: Dataset '/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1' has 20 classes but the model has num_classes=21. Evaluating with the model's head unchanged; class indices may not line up with the dataset.
  results = model.evaluate(
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test/F1          │    0.48650485277175903    │
│         test/loss         │     9.381949424743652     │
│        test/mAP_50        │    0.5013025403022766     │
│      test/mAP_50_95       │    0.27787336707115173    │
│        test/mAP_75        │    0.27045994997024536    │
│         test/mAR          │    0.5276862978935242     │
│      test/precision       │    0.5317834615707397     │
│        test/recall        │    0.48023736476898193    │
└───────────────────────────┴───────────────────────────┘


RAW TEST RESULTS
{'test/loss': 9.381949424743652, 'test/mAP_50_95': 0.27787336707115173, 'test/mAP_50': 0.5013025403022766, 'test/mAP_75': 0.27045994997024536, 'test/mAR': 0.5276862978935242, 'test/F1': 0.48650485277175903, 'test/precision': 0.5317834615707397, 'test/recall': 0.48023736476898193}
